<a href="https://colab.research.google.com/github/elianpuuig/APRENDIZAJE-AUTOM-TICO-1/blob/main/ML_TP2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACHINE LEARNING
**TRABAJOR PRACTICO NUMERO 2  CLUSTERING**

utilizando k-means y DB-SCAN

Utilizando solo Numpy y Scikit-learn, y opcionalmente Pandas y Matplotlib, debes realizar lo siguiente sobre el conjunto de datos de California Housing:
1. Analiza los datos, inspecciona todas las características y el target.
Explica de que trata el dataset, cada una de sus variables

Asegúrate de preparar los datos para utilizarse en modelos predictivos.

Puedes hacer el preprocesamiento de características (selección, transformación) que consideres más adecuado, justificar.

In [3]:
import kagglehub
import os
import pandas as pd

# Descarga la última versión del dataset
path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("Path to dataset files:", path)



files_in_path = os.listdir(path)
print(f"Archivos en el directorio {path}: {files_in_path}")
ruta_al_archivo = path + "/" +files_in_path[0]

print(f"Cargando datos desde : {ruta_al_archivo}")
df = pd.read_csv(ruta_al_archivo)
print("DataFrame 'df' creado. Muestro las primeras 5 filas:")
display(df.head())

Using Colab cache for faster access to the 'california-housing-prices' dataset.
Path to dataset files: /kaggle/input/california-housing-prices
Archivos en el directorio /kaggle/input/california-housing-prices: ['housing.csv']
Cargando datos desde : /kaggle/input/california-housing-prices/housing.csv
DataFrame 'df' creado. Muestro las primeras 5 filas:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [4]:
#DIAGNÓSTICO DE LOS DATOS
print("--- INFO DEL DATASET ---")
df.info()

print("\n--- VALORES NULOS ---")
print(df.isnull().sum())

print("\n--- VALORES ÚNICOS EN VARIABLES CATEGÓRICAS ---")
print(df['ocean_proximity'].value_counts())

--- INFO DEL DATASET ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB

--- VALORES NULOS ---
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income        

***Imputacion***

In [7]:
#Imputacion de Codigo
from sklearn.impute import KNNImputer

#le pasamos las 5 filas con las que va a imputar nuestros datos faltantes
imputadorKNN = KNNImputer(n_neighbors=5)

#utilice fit_transform() que es el clasico metodo para transformar datos en sklearn
#Rellene los huecos y le pasamos los datos en forma 2D
df['total_bedrooms'] = imputadorKNN.fit_transform(df[['total_bedrooms']])

#Verifico la limpieza de nulos en el dataframe
print(df.isnull().sum())

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


***Codificacion***

Utilice la codificacion para pasar valores no numericos a un formato que el algoritmo entienda con la tecnica one hot encoding

In [9]:
#Codificacion de datos, voy a manipular la variable categorica
#usando la tecnica de one hot encoding con (pd.get_dummies())
#le pase dtype=int para que devuelva  1 y 0 en vez de True o False

dfCodificado = pd.get_dummies(df, columns=['ocean_proximity'], dtype=int)

# print(f"Nuevo dataFrame codificado: {dfCodificado}")



***Escalamiento***

voy a utilizar RobustScaler para que utilice la mediana( el valor central, si ordenamos a todos de mayor a menor) y el rango intercuartilico (que va a ser donde viven el 50% de las familias normales) esto hace que logre escalar los datos concentrandose en la masa principal y no en los puntos atipicos u outliers que puedan arruibar nuestro clustering

In [11]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

#Genere un nuevo dataframe y le asigne el nombre original de las columnas,
#para eso le psasamos el scaler que va a transformar los datos de todas las columnas
# de nuestro df previamente codificado para escalarlo
dfScaler = pd.DataFrame(
    scaler.fit_transform(dfCodificado),
    columns=dfCodificado.columns)

#nuestro daataFRame escaladp
dfScaler

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-0.986807,0.957672,0.631579,-0.733422,-0.892419,-0.899787,-0.870769,2.197582,1.880448,0.0,0.0,0.0,1.0,0.0
1,-0.984169,0.952381,-0.421053,2.924276,1.929242,1.316631,2.243077,2.186664,1.232041,0.0,0.0,0.0,1.0,0.0
2,-0.989446,0.949735,1.210526,-0.388178,-0.716245,-0.714286,-0.713846,1.707732,1.187941,0.0,0.0,0.0,1.0,0.0
3,-0.992084,0.949735,1.210526,-0.501691,-0.586282,-0.648188,-0.584615,0.967177,1.113523,0.0,0.0,0.0,1.0,0.0
4,-0.992084,0.949735,1.210526,-0.294074,-0.456318,-0.640725,-0.461538,0.142854,1.119724,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-0.686016,1.380952,-0.210526,-0.271725,-0.184838,-0.342217,-0.243077,-0.905796,-0.700086,0.0,1.0,0.0,0.0,0.0
20636,-0.717678,1.383598,-0.578947,-0.841053,-0.831769,-0.863539,-0.907692,-0.448655,-0.706977,0.0,1.0,0.0,0.0,0.0
20637,-0.720317,1.367725,-0.631579,0.074695,0.135740,-0.169510,0.073846,-0.841709,-0.602239,0.0,1.0,0.0,0.0,0.0
20638,-0.746702,1.367725,-0.578947,-0.157036,-0.083755,-0.453092,-0.184615,-0.765007,-0.654608,0.0,1.0,0.0,0.0,0.0


## 2. Realiza un clustering sobre los datos con K-means, separándolos en un número manejable de  grupos (de 2 a 10).

2.1) Justifica cuál métrica ( o grupo de métricas) utilizas para determinar el valor de k.

2.2) Realiza un análisis descriptivo de cada clúster, grafica para cada uno diagrama cajas.

Selecciono el cluster correcto, para eso tuve que recorrer el rango 2 y 11 y pasarle cada valor k a uestro cluster , entreno el modelo con nuestro dfScaler, extraigo el .inertia_ y lo guardo en la lista de inercis para posteriormente ver que k es la mejor opcion para nuestro cluster o grupo

para visualizarlo utilice la tecnica del codo y grafique utilizando plotly express para un grafico interactivo

In [18]:
#la inercia es la distancia al centroide de cada punto asi que tengo que calcularla
#para que el algoritmo sepa cuantos grupos va a hacer en un rango de 2 y 10
#como dice la consigna, y asi el algoritmo va a saber los n_clusters o grupos va a hacer

from sklearn.cluster import KMeans
import plotly.graph_objects as go

inercia = []

#seleccionamos las inercias entre 2 y 10
for k in range(2,11):
  #se le pasa el numero actual recorrido en ese rango a clauster
  clusteringKmeans = KMeans(n_clusters=k, random_state=0)
  #se entrena el modelo con ese numero y el df previamente scalado, luego extraemos .inertia_
  clusteringKmeans.fit(dfScaler)
  #gaurdamos las inercias de ese rango en nuestra variable inercia
  inercia.append(clusteringKmeans.inertia_)
print(inercia)

#Una vez obtenidas las respecticas inercias graficamos


# Eje X: los valores de K que probamos (del 2 al 10)
valores_k = list(range(2, 11))

#Grfico
grafInercias = go.Figure()
grafInercias.add_trace(go.Scatter(
  x=valores_k,
  y=inercia,
  mode='lines+markers',
  marker=dict(size=10, color='cyan'), line=dict(width=3)))

grafInercias.update_layout(
    title="Método del Codo (Inercia vs K)",
    xaxis_title="Número de Clústers (K)",
    yaxis_title="Inercia (Distancia interna)",
    template="plotly_dark", height=500)

grafInercias.show()

[124138.98049622861, 100669.85142379931, 87370.94078223295, 76893.3866018752, 69915.12498891608, 66121.48768983765, 63302.560963409764, 59523.56165357561, 57192.1186149881]


Una vez seleccionado nuestro cluster ya se cuantos grupos va a hacer en este caso 3 asi que se lo pasamos, y asi se le asignan los grupos a los barrios de california

In [25]:
import plotly.express as px

kmeansDfFinal = KMeans(n_clusters=3, random_state=0)
kmeansDfFinal.fit(dfScaler)
#guardamos con sus respectivas etiquetas de grupo en el df original
#labels_ les pone etiquetas
df['Cluster'] = kmeansDfFinal.labels_
display(df[['median_income', 'population', 'Cluster']].head())

#GRAFICO
#para que plotly no lo tome como una escala continua de colores lo pasamos a txt
df['Cluster'] = df['Cluster'].astype(str)

# 2. Armamos el boxplot del Ingreso Medio para cada grupo
Grafico_uno = px.box(df, x="Cluster", y="median_income", color="Cluster",
             title="Análisis Descriptivo: Ingreso Medio por Clúster",
             labels={"median_income": "Ingreso Medio (Decenas de miles de USD)", "Cluster": "Grupo"},
             template="plotly_dark")

Grafico_uno.show()

Grafico_dos = px.box(df, x="Cluster", y="median_house_value", color="Cluster",
             title="Análisis Descriptivo: Valor medio de las casas por Clúster",
             labels={"median_house_value": "Valor medio de las casas (Decenas de miles de USD)", "Cluster": "Grupo"},
             template="plotly_dark")

Grafico_dos.show()

Grafico_tres = px.box(df, x="Cluster", y="population", color="Cluster",
             title="Análisis Descriptivo: Poblacion por Clúster",
             labels={"population": "Poblacion (Decenas de miles de USD)", "Cluster": "Grupo"},
             template="plotly_dark")

Grafico_tres.show()


,median_income,population,Cluster
0,8.3252,322.0,0
1,8.3014,2401.0,1
2,7.2574,496.0,0
3,5.6431,558.0,0
4,3.8462,565.0,0




## 3. Aplica DB-SCAN sobre el mismo dataset que utilizaste en el punto 2.

Realiza una interpretación de su aplicación y un análisis de los resultados obtenidos.

si le pasamos a nuestro cluster de DB-SCAN un radio (eps) demasiado chico de por ejemplo 0,5 genera muchisimas anomalias ya que tiene un rango muy pequeño a comparar, pero si lo ponemos muy grande ej:1,5 el radio identificaria pocas anomalias , lo mejor es dejarlo en un punto medio 1

In [34]:
from sklearn.cluster import DBSCAN
#eps=0.5 y un min_samples=5
#eps songinifica el radio para encontrar otros puntos
#min_samples el numuero mininmo de puntos para formar un grupo
db_Scan = DBSCAN(eps=1 , min_samples=5)

db_Scan.fit(dfScaler)
etiquetas_dbscan = db_Scan.labels_

serie = pd.Series(etiquetas_dbscan).value_counts()
#Guarde las etiquetas en el DataFrame original (pasadas a texto para colores)
df['DBSCAN_Cluster'] = db_Scan.labels_.astype(str)

#Grafique usando latitud y longitud (¡Se va a dibujar la forma de California!)
graf_mapa = px.scatter(
    df,
    x="longitude", y="latitude",
    color="DBSCAN_Cluster",
    title="Mapa de Clústers y Anomalías (DB-SCAN)",
    labels={"longitude": "Longitud", "latitude": "Latitud"},
    template="plotly_dark",
    opacity=0.6
    )

graf_mapa.show()

# CONCLUSIONES
K-Means es ideal cuando necesitás que cada cliente o barrio pertenezca a un grupo sí o sí (hace particiones globales).


DB-SCAN es ideal cuando buscás patrones naturales y anomalías (ciberseguridad, fraude, barrios extremadamente raros), porque prefiere dejar un dato afuera antes que meterlo a la fuerza en un grupo que no le corresponde.




Lo primero que hice fue analizar los datos, valores nulos y unicos

# **Preparacion de Los datos**

Luego pase a la manipulacion de los datos, primero hice la imputacion de los valores nulos limpiando el dataset de todos esos faltantes con KNNImputer()

luego hice la codificacion pasando a valores numericos las columnas que no lo eran utilizando get_dumnies() (one hote encoding)

y tambien hice un escalamiento de los datos para poder concentrarme en la media de los datos y no en los valores atipicos para que el algoritmo sea mucho mas especifico

#**Aplicacion del modelo**
Una vez los datos manipulados y posteriormente limpios puedo empezar a aplicar el modelo